In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles, make_moons
# # Create a synthetic multi-domain dataset
# def create_synthetic_data():
#     # Define centers for four domains, each with mean shift
#     centers = [
#         [(-5, -5), (5, 5)],  # Domain 1
#         [(-5, 5), (5, -5)],  # Domain 2
#         [(0, -5), (0, 5)],   # Domain 3
#         [(-5, 0), (5, 0)]    # Domain 4
#     ]

#     domain_data = []
#     domain_labels = []

#     for domain_idx, domain_centers in enumerate(centers):
#         X, y = make_blobs(n_samples=500, centers=domain_centers, cluster_std=1.0, random_state=domain_idx)
#         domain_data.append(X)
#         domain_labels.append(y)

#     return domain_data, domain_labels
def create_synthetic_data():
    domain_data = []
    domain_labels = []

    # Base data: Circles
    base_X, base_y = make_circles(n_samples=1000, factor=0.1, noise=0.1, random_state=1)

    # Domain 1: Original data
    domain_data.append(base_X)
    domain_labels.append(base_y)

    # Domain 2: Scaled data
    scaled_X = base_X * np.array([1.5, 0.8])  # Stretch in x and compress in y
    domain_data.append(scaled_X)
    domain_labels.append(base_y)

    # Domain 3: Rotated data
    rotation_matrix = np.array([[np.cos(np.pi / 4), -np.sin(np.pi / 4)], 
                                 [np.sin(np.pi / 4), np.cos(np.pi / 4)]])
    rotated_X = base_X @ rotation_matrix.T
    domain_data.append(rotated_X)
    domain_labels.append(base_y)

    # Domain 4: Translated data
    translated_X = base_X + np.array([2, -1])  # Shift in x and y
    domain_data.append(translated_X)
    domain_labels.append(base_y)

    return domain_data, domain_labels


def plot_decision_boundary(model, X, y, domain_labels, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                         np.arange(y_min, y_max, 0.1))
    grid = np.c_[xx.ravel(), yy.ravel()]
    
    with torch.no_grad():
        preds = model(torch.tensor(grid, dtype=torch.float32)).argmax(dim=1).numpy()

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, preds.reshape(xx.shape), alpha=0.8, cmap=plt.cm.coolwarm)
    plt.scatter(X[:, 0], X[:, 1], c=y, s=40, cmap=plt.cm.coolwarm, edgecolor='k')
    plt.title(title)
    plt.show()

def plot_feature_space(features, labels, domains, title):
    tsne = TSNE(n_components=2, random_state=42)
    reduced_features = tsne.fit_transform(features)
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(reduced_features[:, 0], reduced_features[:, 1], c=labels, cmap=plt.cm.coolwarm, alpha=0.8)
    plt.title(title)
    plt.colorbar(scatter, label='Class')
    plt.show()

def plot_kernel_matrix(kernel_matrix, title):
    plt.figure(figsize=(8, 6))
    plt.imshow(kernel_matrix, cmap='viridis', interpolation='nearest')
    plt.colorbar()
    plt.title(title)
    plt.show()

# Define a simple MLP model
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)


In [2]:

# Train the model
def train_model(model, X_train, y_train, alpha=0.5, use_dpp=False, gamma=0.1):
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    epochs = 100

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        
        features = model.fc1(torch.tensor(X_train, dtype=torch.float32))
        logits = model(torch.tensor(X_train, dtype=torch.float32))
        
        ce_loss = F.cross_entropy(logits, torch.tensor(y_train, dtype=torch.long))
        if use_dpp:
            kernel_matrix = rbf_kernel(features.detach().cpu().numpy(), gamma=gamma)
            diversity_loss = -torch.logdet(torch.tensor(kernel_matrix))
            total_loss = alpha * ce_loss + (1 - alpha) * diversity_loss
        else:
            total_loss = ce_loss

        total_loss.backward()
        optimizer.step()
    return model

def eval_model(model, X_test, y_test):
    # Evaluate
    model.eval()
    with torch.no_grad():
        test_preds = model(torch.tensor(X_test, dtype=torch.float32)).argmax(dim=1).numpy()
        test_accs = accuracy_score(y_test, test_preds)

    return test_accs



In [3]:

# Leave-one-domain-out evaluation
def leave_one_domain_out_evaluation(domain_data, domain_labels, alpha=0.5, use_dpp=False, gamma=0.1):
    num_domains = len(domain_data)
    test_accuracies = []

    for test_domain in range(num_domains):
        # Prepare train and test data
        X_train = np.vstack([domain_data[i] for i in range(num_domains) if i != test_domain])
        y_train = np.hstack([domain_labels[i] for i in range(num_domains) if i != test_domain])
        X_test = domain_data[test_domain]
        y_test = domain_labels[test_domain]
        
        print(f"Training with domains: {[i for i in range(num_domains) if i != test_domain]}")
        print(f"Testing for domain: {test_domain}")

        model = MLP(input_dim=2, hidden_dim=16, output_dim=2)
        trained_model = train_model(
            model, X_train, y_train, alpha=alpha, use_dpp=use_dpp, gamma=gamma
        )
        test_accs = eval_model(trained_model, X_test, y_test)
        print(f"Accuracy is: {test_accs}")
        test_accuracies.append(test_accs)
        
    average_accuracy = np.mean(test_accuracies)
    return test_accuracies, average_accuracy



In [4]:

# Main function
def main():
    # Data preparation
    domain_data, domain_labels = create_synthetic_data()    # Leave-one-domain-out evaluation
    
    # use dpp
    test_accuracies, average_accuracy = leave_one_domain_out_evaluation(domain_data, domain_labels, alpha=0.5, use_dpp=True, gamma=0.01)
    
    # no dpp
    test_accuracies1, average_accuracy1 = leave_one_domain_out_evaluation(domain_data, domain_labels, alpha=0.5, use_dpp=False, gamma=0.01)

    print(f"Test accuracies for each domain (ERMDPP): {test_accuracies}")
    print(f"Average test accuracy (ERMDPP): {average_accuracy:.4f}")
    
    print(f"Test accuracies for each domain (ERM): {test_accuracies1}")
    print(f"Average test accuracy (ERM): {average_accuracy1:.4f}")
    

In [5]:
if __name__ == "__main__":
    main()

Training with domains: [1, 2, 3]
Testing for domain: 0
Accuracy is: 0.973
Training with domains: [0, 2, 3]
Testing for domain: 1
Accuracy is: 0.967
Training with domains: [0, 1, 3]
Testing for domain: 2
Accuracy is: 0.95
Training with domains: [0, 1, 2]
Testing for domain: 3
Accuracy is: 0.5
Training with domains: [1, 2, 3]
Testing for domain: 0
Accuracy is: 0.967
Training with domains: [0, 2, 3]
Testing for domain: 1
Accuracy is: 0.95
Training with domains: [0, 1, 3]
Testing for domain: 2
Accuracy is: 0.976
Training with domains: [0, 1, 2]
Testing for domain: 3
Accuracy is: 0.5
Test accuracies for each domain (ERMDPP): [0.973, 0.967, 0.95, 0.5]
Average test accuracy (ERMDPP): 0.8475
Test accuracies for each domain (ERM): [0.967, 0.95, 0.976, 0.5]
Average test accuracy (ERM): 0.8482
